# 03 - Cleaning Verification & Sanity Audit

This notebook audits the output of `src/clean.py` against the raw interim extraction (`2024-01`).
It validates:
1. Dataset integrity, row attrition, and duplicate elimination
2. Chronological assertions ($t_{req} \le t_{pickup} < t_{dropoff}$)
3. Operational and telematics boundaries (duration, distance, implied speed)
4. Economic sanity (positive base fare and driver pay)
5. Volume distributions across all target venue and spillover zones

In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

# Path resolution for interim raw vs cleaned
INTERIM_DIR = Path('../data/interim') if Path('../data/interim').exists() else Path('data/interim')

raw_path = INTERIM_DIR / 'venue_trips_2024-01.parquet'
clean_path = INTERIM_DIR / 'cleaned_venue_trips_2024-01.parquet'

assert raw_path.exists(), f'Missing raw interim file: {raw_path}'
assert clean_path.exists(), f'Missing cleaned file: {clean_path}'

df_raw = pd.read_parquet(raw_path)
df_clean = pd.read_parquet(clean_path)

print(f'Raw Interim Records:   {len(df_raw):,}')
print(f'Cleaned Records:       {len(df_clean):,}')
print(f'Records Filtered Out:  {len(df_raw) - len(df_clean):,} ({(len(df_raw) - len(df_clean)) / len(df_raw) * 100:.2f}%)')
print(f'Overall Retention:     {len(df_clean) / len(df_raw) * 100:.2f}%')

Raw Interim Records:   19,663,930
Cleaned Records:       19,460,188
Records Filtered Out:  203,742 (1.04%)
Overall Retention:     98.96%


### 1. Deduplication and Null Audits

In [3]:
raw_dupes = df_raw.duplicated().sum()
clean_dupes = df_clean.duplicated().sum()
clean_nulls = df_clean.isna().sum().sum()

print(f'Raw exact duplicates:      {raw_dupes:,}')
print(f'Clean exact duplicates:    {clean_dupes:,} (Expected: 0)')
print(f'Clean total null values:   {clean_nulls:,} (Expected: 0)')

assert clean_dupes == 0, 'Cleaned dataset contains unexpected duplicates!'
assert clean_nulls == 0, 'Cleaned dataset contains unexpected nulls!'

Raw exact duplicates:      0
Clean exact duplicates:    0 (Expected: 0)
Clean total null values:   10,244,990 (Expected: 0)


AssertionError: Cleaned dataset contains unexpected nulls!

### 2. Chronological Integrity
Verifies: $\text{request\_datetime} \le \text{pickup\_datetime} < \text{dropoff\_datetime}$

In [4]:
req_after_pickup = (df_clean['request_datetime'] > df_clean['pickup_datetime']).sum()
pickup_after_dropoff = (df_clean['pickup_datetime'] >= df_clean['dropoff_datetime']).sum()

print(f'Violations (request > pickup):     {req_after_pickup}')
print(f'Violations (pickup >= dropoff):    {pickup_after_dropoff}')

assert req_after_pickup == 0, 'Found trips requested after pickup!'
assert pickup_after_dropoff == 0, 'Found trips picked up after or at dropoff!'

Violations (request > pickup):     0
Violations (pickup >= dropoff):    0


### 3. Operational & Telematics Bounds
Checking threshold compliance:
- $60\text{ s} \le \text{trip\_time} \le 14,400\text{ s}$
- $0.1\text{ mi} \le \text{trip\_miles} \le 100.0\text{ mi}$
- $\text{implied\_speed} \le 75.0\text{ mph}$

In [5]:
implied_speed = df_clean['trip_miles'] / (df_clean['trip_time'] / 3600.0)

bounds_audit = pd.DataFrame({
    'Metric': ['trip_time (sec)', 'trip_miles', 'implied_speed (mph)'],
    'Min': [df_clean['trip_time'].min(), df_clean['trip_miles'].min(), implied_speed.min()],
    'Median': [df_clean['trip_time'].median(), df_clean['trip_miles'].median(), implied_speed.median()],
    'Max': [df_clean['trip_time'].max(), df_clean['trip_miles'].max(), implied_speed.max()],
    'Allowed Min': [60, 0.1, 0.0],
    'Allowed Max': [14400, 100.0, 75.0]
})
bounds_audit.round(2)

,Metric,Min,Median,Max,Allowed Min,Allowed Max
0,trip_time (sec),60.00,907.00,14345.00,60.0,14400.0
1,trip_miles,0.10,2.81,99.98,0.1,100.0
2,implied_speed (mph),0.06,11.57,74.93,0.0,75.0


### 4. Economic Bounds
Verifies: $\text{base\_passenger\_fare} \ge 0.01$ and $\text{driver\_pay} \ge 0.00$

In [6]:
invalid_fares = (df_clean['base_passenger_fare'] <= 0).sum()
invalid_pay = (df_clean['driver_pay'] < 0).sum()

print(f'Trips with non-positive base fare: {invalid_fares}')
print(f'Trips with negative driver pay:     {invalid_pay}')

assert invalid_fares == 0, 'Found non-positive fares!'
assert invalid_pay == 0, 'Found negative driver pay!'

Trips with non-positive base fare: 0
Trips with negative driver pay:     0


### 5. Zone Volume Attrition Comparison
Verifies that cleaning did not disproportionately drop records from any single venue or spillover zone.

In [8]:
zone_labels = {
    189: 'Barclays Core (189)',
    49: 'Barclays Spillover (49)',
    61: 'Barclays Spillover (61)',
    62: 'Barclays Spillover (62)',
    97: 'Barclays Spillover (97)',
    181: 'Barclays Spillover (181)',
    190: 'Barclays Spillover (190)',
    253: 'Citi Field Core (253)',
    92: 'Citi Field Spillover (92)',
    93: 'Citi Field Spillover (93)'
}

raw_counts = df_raw.loc[df_raw['PULocationID'].isin(zone_labels), 'PULocationID'].value_counts()
clean_counts = df_clean.loc[df_clean['PULocationID'].isin(zone_labels), 'PULocationID'].value_counts()


comparison = pd.DataFrame({
    'Zone Name': pd.Series(zone_labels),
    'Raw Trips': raw_counts,
    'Clean Trips': clean_counts
}).fillna(0)

comparison['Dropped'] = comparison['Raw Trips'] - comparison['Clean Trips']
comparison['Retention (%)'] = (comparison['Clean Trips'] / comparison['Raw Trips'] * 100).round(2)
comparison.sort_index()

,Zone Name,Raw Trips,Clean Trips,Dropped,Retention (%)
49,Barclays Spillover (49),117734,116558,1176,99.00
61,Barclays Spillover (61),256556,253564,2992,98.83
62,Barclays Spillover (62),72028,71216,812,98.87
92,Citi Field Spillover (92),97103,95696,1407,98.55
93,Citi Field Spillover (93),5350,5314,36,99.33
97,Barclays Spillover (97),121814,120495,1319,98.92
181,Barclays Spillover (181),191073,189255,1818,99.05
189,Barclays Core (189),73051,72353,698,99.04
190,Barclays Spillover (190),8460,8412,48,99.43
253,Citi Field Core (253),430,428,2,99.53
